In [1]:
# ============================================================
# MINING-BTC-180.CSV
# NLP + TF-IDF + K-MEANS CLUSTERING
# Complete Jupyter Notebook Code
# ============================================================


# ============================================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# ============================================================

# Run this cell if the libraries are not already installed.

# !pip install pandas numpy matplotlib seaborn scikit-learn nltk


# ============================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings("ignore")

# NLP
import nltk
from nltk.corpus import stopwords

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Download NLTK stopwords
nltk.download("stopwords")


# ============================================================
# CELL 3: LOAD DATASET
# ============================================================

file_name = "Mining-BTC-180.csv"

try:
    df = pd.read_csv(file_name)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"ERROR: {file_name} was not found.")
    print("Make sure the CSV file is in the same folder as this Jupyter Notebook.")
    raise

print("\nDataset shape:")
print(df.shape)

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# CELL 4: DATASET INFORMATION
# ============================================================

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nDataset dimensions:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


# ============================================================
# CELL 5: CHECK MISSING VALUES
# ============================================================

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())


# ============================================================
# CELL 6: REMOVE COMPLETELY EMPTY ROWS
# ============================================================

df = df.dropna(how="all").reset_index(drop=True)

print("Dataset shape after removing empty rows:")
print(df.shape)


# ============================================================
# CELL 7: IDENTIFY TEXT COLUMNS
# ============================================================

# Find columns containing text/string data
text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("\nText columns detected:")
for column in text_columns:
    print("-", column)

if len(text_columns) == 0:
    raise ValueError(
        "No text columns were detected. "
        "K-Means NLP requires at least one text column."
    )


# ============================================================
# CELL 8: SHOW UNIQUE VALUES AND DATA TYPES
# ============================================================

print("\nColumn data types:")
print(df.dtypes)

print("\nNumber of unique values:")
for column in df.columns:
    print(f"{column}: {df[column].nunique()}")


# ============================================================
# CELL 9: FILL MISSING TEXT VALUES
# ============================================================

# Replace missing values in text columns with empty strings
for column in text_columns:
    df[column] = df[column].fillna("")

print("\nMissing values after cleaning text columns:")
print(df[text_columns].isnull().sum())


# ============================================================
# CELL 10: COMBINE TEXT COLUMNS
# ============================================================

# Combine all detected text columns into one text field

df["combined_text"] = df[text_columns].apply(
    lambda row: " ".join(
        str(value) for value in row
        if str(value).strip() != ""
    ),
    axis=1
)

print("\nCombined text examples:")
display(df[["combined_text"]].head(10))


# ============================================================
# CELL 11: CHECK EMPTY TEXT RECORDS
# ============================================================

empty_text = (df["combined_text"].str.strip() == "").sum()

print("Number of records with empty text:", empty_text)

# Remove records that have no text
df = df[df["combined_text"].str.strip() != ""].reset_index(drop=True)

print("Dataset shape after removing empty text:")
print(df.shape)


# ============================================================
# CELL 12: NLP PREPROCESSING
# ============================================================

stop_words = set(stopwords.words("english"))

def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    # Remove numbers
    # Comment this line if numbers are important in your dataset
    text = re.sub(
        r"\b\d+\b",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Remove stopwords
    words = [
        word
        for word in text.split()
        if word not in stop_words
    ]

    return " ".join(words)


# Apply NLP preprocessing
df["clean_text"] = df["combined_text"].apply(preprocess_text)

print("\nOriginal and cleaned text:")
display(
    df[
        ["combined_text", "clean_text"]
    ].head(10)
)


# ============================================================
# CELL 13: REMOVE RECORDS WITH EMPTY CLEANED TEXT
# ============================================================

df = df[
    df["clean_text"].str.strip() != ""
].reset_index(drop=True)

print("Dataset shape after NLP cleaning:")
print(df.shape)


# ============================================================
# CELL 14: TF-IDF VECTORIZATION
# ============================================================

# TF-IDF converts text into numerical values.

tfidf = TfidfVectorizer(
    max_features=2000,
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = tfidf.fit_transform(
    df["clean_text"]
)

print("TF-IDF completed.")

print("\nTF-IDF matrix shape:")
print(X.shape)

print(
    "\nNumber of TF-IDF features:",
    len(tfidf.get_feature_names_out())
)


# ============================================================
# CELL 15: DISPLAY TF-IDF FEATURES
# ============================================================

feature_names = tfidf.get_feature_names_out()

print("\nFirst 50 TF-IDF features:")

for i, feature in enumerate(feature_names[:50]):
    print(i + 1, feature)


# ============================================================
# CELL 16: DISPLAY TF-IDF MATRIX
# ============================================================

# Convert only a small part of the matrix to a DataFrame
# so that it is easier to inspect.

sample_size = min(10, X.shape[0])
feature_sample = min(20, X.shape[1])

tfidf_sample = pd.DataFrame(
    X[:sample_size, :feature_sample].toarray(),
    columns=feature_names[:feature_sample]
)

print("Sample TF-IDF matrix:")
display(tfidf_sample)


# ============================================================
# CELL 17: ELBOW METHOD
# ============================================================

inertias = []

# Test K values
max_k = min(10, len(df) - 1)

k_values = range(2, max_k + 1)

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X)

    inertias.append(
        kmeans.inertia_
    )


# Plot elbow curve
plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.xticks(list(k_values))

plt.grid(True)

plt.show()


# ============================================================
# CELL 18: SILHOUETTE SCORE
# ============================================================

silhouette_scores = []

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(score)

    print(
        f"K = {k}, "
        f"Silhouette Score = {score:.4f}"
    )


# Plot silhouette scores
plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")

plt.title(
    "Silhouette Score for Different K Values"
)

plt.xticks(list(k_values))

plt.grid(True)

plt.show()


# ============================================================
# CELL 19: AUTOMATICALLY SELECT BEST K
# ============================================================

best_k_index = np.argmax(
    silhouette_scores
)

best_k = list(k_values)[best_k_index]

best_score = silhouette_scores[best_k_index]

print("\nBest K based on Silhouette Score:")
print("K =", best_k)

print(
    "Silhouette Score =",
    round(best_score, 4)
)


# ============================================================
# CELL 20: RUN FINAL K-MEANS
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20,
    max_iter=300
)

df["cluster"] = kmeans.fit_predict(X)

print("K-Means clustering completed.")


# ============================================================
# CELL 21: CLUSTER COUNTS
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\nNumber of records in each cluster:")

print(cluster_counts)


# ============================================================
# CELL 22: CLUSTER DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Records")

plt.title(
    "Number of Records in Each K-Means Cluster"
)

plt.show()


# ============================================================
# CELL 23: IMPORTANT WORDS FOR EACH CLUSTER
# ============================================================

terms = tfidf.get_feature_names_out()

centroids = kmeans.cluster_centers_

print("\nTOP WORDS FOR EACH CLUSTER")
print("=" * 70)

cluster_keywords = {}

for cluster_number in range(best_k):

    # Get indices of highest centroid values
    order = centroids[
        cluster_number
    ].argsort()[::-1]

    # Get top 15 words
    top_words = [
        terms[index]
        for index in order[:15]
    ]

    cluster_keywords[
        cluster_number
    ] = top_words

    print(
        f"\nCluster {cluster_number}"
    )

    print(
        "Top words:"
    )

    print(
        ", ".join(top_words)
    )


# ============================================================
# CELL 24: CREATE KEYWORD TABLE
# ============================================================

keyword_table = pd.DataFrame(
    dict(
        (
            cluster,
            pd.Series(words)
        )
        for cluster, words
        in cluster_keywords.items()
    )
)

keyword_table.index = [
    f"Top Word {i + 1}"
    for i in range(
        keyword_table.shape[0]
    )
]

print("\nCluster keyword table:")

display(keyword_table)


# ============================================================
# CELL 25: DISPLAY RECORDS FROM EACH CLUSTER
# ============================================================

for cluster_number in sorted(
    df["cluster"].unique()
):

    print("\n")
    print("=" * 80)
    print(
        f"CLUSTER {cluster_number}"
    )
    print("=" * 80)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    print(
        "Number of records:",
        len(cluster_data)
    )

    # Display up to 5 examples
    display(
        cluster_data[
            text_columns +
            ["clean_text", "cluster"]
        ].head(5)
    )


# ============================================================
# CELL 26: PCA DIMENSIONALITY REDUCTION
# ============================================================

# PCA is used only for visualisation.

# Convert sparse matrix to dense matrix
X_dense = X.toarray()

print(
    "Original feature dimensions:",
    X_dense.shape
)

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_dense
)

print(
    "PCA dimensions:",
    X_pca.shape
)

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_
)


# ============================================================
# CELL 27: CREATE PCA DATAFRAME
# ============================================================

pca_df = pd.DataFrame(
    X_pca,
    columns=[
        "PCA1",
        "PCA2"
    ]
)

pca_df["cluster"] = df[
    "cluster"
].values

display(
    pca_df.head()
)


# ============================================================
# CELL 28: VISUALISE K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=pca_df,
    x="PCA1",
    y="PCA2",
    hue="cluster",
    palette="viridis",
    s=80,
    alpha=0.8
)

plt.title(
    "K-Means Clusters Visualised Using PCA"
)

plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)

plt.legend(
    title="Cluster"
)

plt.grid(True)

plt.show()


# ============================================================
# CELL 29: PLOT CLUSTER CENTRES
# ============================================================

# Transform K-Means cluster centres into PCA space

centres_pca = pca.transform(
    centroids
)

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=pca_df,
    x="PCA1",
    y="PCA2",
    hue="cluster",
    palette="viridis",
    s=60,
    alpha=0.6
)

plt.scatter(
    centres_pca[:, 0],
    centres_pca[:, 1],
    marker="X",
    s=300,
    label="Cluster Centres"
)

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.title(
    "K-Means Clusters and Cluster Centres"
)

plt.legend()

plt.grid(True)

plt.show()


# ============================================================
# CELL 30: FINAL SILHOUETTE SCORE
# ============================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print(
    "Final Silhouette Score:",
    round(final_silhouette, 4)
)

if final_silhouette >= 0.7:

    print(
        "Interpretation: Strong cluster separation."
    )

elif final_silhouette >= 0.5:

    print(
        "Interpretation: Reasonable cluster separation."
    )

elif final_silhouette >= 0.25:

    print(
        "Interpretation: Weak/moderate cluster separation."
    )

else:

    print(
        "Interpretation: Poor cluster separation."
    )


# ============================================================
# CELL 31: CLUSTER SUMMARY
# ============================================================

summary_data = []

for cluster_number in sorted(
    df["cluster"].unique()
):

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    summary_data.append({

        "Cluster":
            cluster_number,

        "Number of Records":
            len(cluster_data),

        "Percentage":
            round(
                len(cluster_data)
                / len(df)
                * 100,
                2
            ),

        "Top Keywords":
            ", ".join(
                cluster_keywords[
                    cluster_number
                ][:10]
            )
    })


cluster_summary = pd.DataFrame(
    summary_data
)

print("\nCLUSTER SUMMARY")

display(
    cluster_summary
)


# ============================================================
# CELL 32: ADD PCA VALUES TO ORIGINAL DATASET
# ============================================================

df["PCA1"] = X_pca[:, 0]

df["PCA2"] = X_pca[:, 1]

print(
    "PCA values added to dataset."
)

display(
    df.head()
)


# ============================================================
# CELL 33: SORT DATA BY CLUSTER
# ============================================================

df_sorted = df.sort_values(
    by="cluster"
).reset_index(
    drop=True
)

display(
    df_sorted.head(20)
)


# ============================================================
# CELL 34: SAVE CLUSTERED DATASET
# ============================================================

output_file = (
    "Mining-BTC-180_clustered.csv"
)

df_sorted.to_csv(
    output_file,
    index=False
)

print(
    "Clustered dataset saved successfully:"
)

print(
    output_file
)


# ============================================================
# CELL 35: SAVE CLUSTER KEYWORDS
# ============================================================

keyword_file = (
    "Mining-BTC-180_cluster_keywords.csv"
)

keyword_table.to_csv(
    keyword_file
)

print(
    "Cluster keyword file saved:"
)

print(
    keyword_file
)


# ============================================================
# CELL 36: FINAL RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("FINAL K-MEANS NLP RESULTS")
print("=" * 80)

print(
    f"Original records: {len(df)}"
)

print(
    f"Text columns used: {text_columns}"
)

print(
    f"TF-IDF features: {X.shape[1]}"
)

print(
    f"Best number of clusters: {best_k}"
)

print(
    f"Final silhouette score: "
    f"{final_silhouette:.4f}"
)

print("\nCluster sizes:")

for cluster_number, count in cluster_counts.items():

    percentage = (
        count / len(df) * 100
    )

    print(
        f"Cluster {cluster_number}: "
        f"{count} records "
        f"({percentage:.2f}%)"
    )

print("\nTop keywords:")

for cluster_number in range(best_k):

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(
            cluster_keywords[
                cluster_number
            ]
        )
    )

print("\n")
print(
    "Output files:"
)

print(
    "- Mining-BTC-180_clustered.csv"
)

print(
    "- Mining-BTC-180_cluster_keywords.csv"
)

print("=" * 80)

Dataset loaded successfully.

Dataset shape:
(179, 9)

First 5 rows:


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Unnamed: 0,Date,Number-transactions,Output-volume(BTC),Market-price,Hash-rate,Cost-per-trans-USD,Mining-revenue-USD,Transaction-fees-BTC
0,0,2017-04-29 00:00:00,341319,4488916,3119179,4488916,9,3119179,256
1,1,2017-04-30 00:00:00,281489,3918072,2720216,3918072,10,2720216,199
2,2,2017-05-01 00:00:00,294786,3892124,2878278,3892124,10,2878278,228
3,3,2017-05-02 00:00:00,333161,4099704,3149553,4099704,10,3149553,273
4,4,2017-05-03 00:00:00,295149,3425069,2760373,3425069,10,2760373,247



Column names:
['Unnamed: 0', 'Date', 'Number-transactions', 'Output-volume(BTC)', 'Market-price', 'Hash-rate', 'Cost-per-trans-USD', 'Mining-revenue-USD', 'Transaction-fees-BTC']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Unnamed: 0            179 non-null    int64
 1   Date                  179 non-null    str  
 2   Number-transactions   179 non-null    int64
 3   Output-volume(BTC)    179 non-null    int64
 4   Market-price          179 non-null    int64
 5   Hash-rate             179 non-null    int64
 6   Cost-per-trans-USD    179 non-null    int64
 7   Mining-revenue-USD    179 non-null    int64
 8   Transaction-fees-BTC  179 non-null    int64
dtypes: int64(8), str(1)
memory usage: 16.0 KB

Dataset dimensions:
Rows: 179
Columns: 9

Missing values in each column:
Unnamed: 0              0
Date               

,combined_text
0,2017-04-29 00:00:00
1,2017-04-30 00:00:00
2,2017-05-01 00:00:00
3,2017-05-02 00:00:00
4,2017-05-03 00:00:00
5,2017-05-04 00:00:00
6,2017-05-05 00:00:00
7,2017-05-06 00:00:00
8,2017-05-07 00:00:00
9,2017-05-08 00:00:00


Number of records with empty text: 0
Dataset shape after removing empty text:
(179, 10)

Original and cleaned text:


,combined_text,clean_text
0,2017-04-29 00:00:00,
1,2017-04-30 00:00:00,
2,2017-05-01 00:00:00,
3,2017-05-02 00:00:00,
4,2017-05-03 00:00:00,
5,2017-05-04 00:00:00,
6,2017-05-05 00:00:00,
7,2017-05-06 00:00:00,
8,2017-05-07 00:00:00,
9,2017-05-08 00:00:00,


Dataset shape after NLP cleaning:
(0, 11)


ValueError: empty vocabulary; perhaps the documents only contain stop words